# DDIM Latent Space Interpolation
Sample a man and woman from SD, then interpolate between them in noise space.

In [ ]:
!pip install -q diffusers transformers accelerate

In [ ]:
import torch, gc, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline, DDIMScheduler

device = "cuda"
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16, safety_checker=None
)
pipe.scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")
pipe.enable_sequential_cpu_offload()   # keeps VRAM usage low
pipe.enable_attention_slicing(1)

vae, unet = pipe.vae, pipe.unet
tokenizer, text_enc = pipe.tokenizer, pipe.text_encoder
scheduler = pipe.scheduler
print("Loaded ✓")

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

@torch.no_grad()
def get_emb(prompt):
    tok = tokenizer(prompt, return_tensors="pt", padding="max_length",
                    max_length=77, truncation=True).input_ids.to("cpu")
    return text_enc(tok)[0].half()

@torch.no_grad()
def decode(latent):
    lat = latent.to(device, dtype=torch.float16)
    img = vae.decode(lat / 0.18215).sample
    img = (img.clamp(-1, 1) + 1) / 2
    arr = img.squeeze(0).permute(1, 2, 0).cpu().float().numpy()
    return Image.fromarray((arr * 255).astype(np.uint8))

def slerp(t, a, b):
    af, bf = a.float().flatten(), b.float().flatten()
    dot = torch.dot(af / af.norm(), bf / bf.norm()).clamp(-1, 1)
    if abs(dot.item()) > 0.9995:
        return (1 - t) * a + t * b
    theta = torch.acos(dot)
    return (torch.sin((1-t)*theta)/torch.sin(theta))*a + \
           (torch.sin(   t *theta)/torch.sin(theta))*b

@torch.no_grad()
def generate(prompt, seed, num_steps=30):
    """Generate an image and return (noise, final_latent)."""
    scheduler.set_timesteps(num_steps)
    emb    = get_emb(prompt).to(device)
    uncond = get_emb("").to(device)

    # Sample starting noise
    g = torch.Generator(device).manual_seed(seed)
    latent = torch.randn((1, 4, 64, 64), generator=g,
                          dtype=torch.float16, device=device)
    latent = latent * scheduler.init_noise_sigma
    noise  = latent.clone().cpu()   # ← this is what we interpolate

    for t in scheduler.timesteps:
        # Run uncond and cond separately (avoids OOM from batch doubling)
        u = unet(latent, t, encoder_hidden_states=uncond).sample
        c = unet(latent, t, encoder_hidden_states=emb).sample
        pred   = u + 7.5 * (c - u)   # classifier-free guidance
        latent = scheduler.step(pred, t, latent).prev_sample

    return noise, latent.cpu()

print("Helpers defined ✓")

In [ ]:
# ── Generate man and woman ────────────────────────────────────────────────────
gc.collect(); torch.cuda.empty_cache()

noise_a, lat_a = generate("portrait of a man, photorealistic, sharp",   seed=42)
gc.collect(); torch.cuda.empty_cache()

noise_b, lat_b = generate("portrait of a woman, photorealistic, sharp", seed=99)
gc.collect(); torch.cuda.empty_cache()

# Show the two endpoints
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(decode(lat_a)); axes[0].set_title("Man  (α=0)");   axes[0].axis("off")
axes[1].imshow(decode(lat_b)); axes[1].set_title("Woman (α=1)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── Interpolate in noise space + decode each frame ────────────────────────────
NUM_FRAMES = 7
emb_a = get_emb("portrait of a man, photorealistic, sharp")
emb_b = get_emb("portrait of a woman, photorealistic, sharp")

frames = []
for i, alpha in enumerate(np.linspace(0, 1, NUM_FRAMES)):
    print(f"Frame {i+1}/{NUM_FRAMES}  α={alpha:.2f}")

    # Slerp the starting noise
    interp_noise = slerp(alpha, noise_a, noise_b).to(device, dtype=torch.float16)
    interp_noise = interp_noise * scheduler.init_noise_sigma

    # Slerp the text conditioning
    interp_emb = slerp(alpha, emb_a, emb_b).to(device)
    uncond     = get_emb("").to(device)

    # Denoise from interpolated noise
    scheduler.set_timesteps(30)
    lat = interp_noise.clone()
    with torch.no_grad():
        for t in scheduler.timesteps:
            u = unet(lat, t, encoder_hidden_states=uncond).sample
            c = unet(lat, t, encoder_hidden_states=interp_emb).sample
            pred = u + 7.5 * (c - u)
            lat  = scheduler.step(pred, t, lat).prev_sample

    frames.append(decode(lat))
    torch.cuda.empty_cache()

print("Done ✓")

In [ ]:
# ── Display strip ─────────────────────────────────────────────────────────────
alphas = np.linspace(0, 1, NUM_FRAMES)
fig, axes = plt.subplots(1, NUM_FRAMES, figsize=(3*NUM_FRAMES, 3))
for ax, frame, a in zip(axes, frames, alphas):
    ax.imshow(frame); ax.set_title(f"α={a:.2f}"); ax.axis("off")
plt.suptitle("DDIM Noise Interpolation: Man → Woman")
plt.tight_layout()
plt.savefig("interpolation_strip.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Save GIF ──────────────────────────────────────────────────────────────────
gif = frames + frames[-2:0:-1]
gif[0].save("interpolation.gif", save_all=True, append_images=gif[1:],
            duration=200, loop=0)
from IPython.display import Image as IPImage
IPImage("interpolation.gif")